# Learning to Rank on MSLR-WEB10k: A Worked Exercise

## Intro

This notebook walks through a small end-to-end learning-to-rank (LTR) workflow on a 100-query sample of the MSLR-WEB10k dataset. The goal is not to build a state-of-the-art ranker, but to show the reasoning behind each step:

- Data preparation: query-level train/test splitting to avoid leakage, and min-max scaling fit on train only
- Evaluation: NDCG@10 implemented from scratch (with unit tests), plus MSE for comparison
- Baselines: random, best single feature, and predict-the-mean, so every model has a reference point
- Models: a pointwise approach (linear regression, ridge) and a pairwise approach (linear RankNet trained with plain SGD)
- Analysis: why the metrics behave differently across approaches, and what the results do and don’t tell us

Each section includes short notes on why a choice was made, not just what the code does.

## Data load

We use the MSLR-WEB10k dataset: web search queries, each with a list of documents. Every document has 136 features and a relevance label from 0 to 4.

The data comes as one row per query, so we convert it to one row per (query, document) pair. We keep only the first N_QUERIES queries to keep things fast.

So each row contains the document_id, query_id, numerical features for that pair (these can be different varieties of lexical (e.g. BM25) and semantic (e.g. embeddings) similarities between the two), and the relevance label (that can be manually annotated by humans).

In [1]:
import sys
!{sys.executable} -m pip install -q pandas pyarrow numpy

import pandas as pd
import numpy as np

# --- Step 1: pull a chunk of queries (not the whole file) ---
N_QUERIES = 100  # how many queries' worth of data to pull -- tune as needed

url = "https://huggingface.co/datasets/philipphager/MSLR-WEB10k/resolve/refs%2Fconvert%2Fparquet/default/train/0000.parquet"
df = pd.read_parquet(url)          # this still reads the whole parquet file first,
df = df.head(N_QUERIES)            # then we just keep a chunk of queries to work with

# --- Step 2: explode from "one row per query" to "one row per (query, doc)" ---
# This is the shape libsvm/load_svmlight_file gives you: flat rows, one per
# query-document pair, with a parallel qid array.
flat = df.explode(["labels", "features"]).reset_index(drop=True)

qid = flat["query"].to_numpy()
y = flat["labels"].to_numpy(dtype=float)
X = np.vstack(flat["features"].to_numpy())  # dense (n_docs, 136) array

print("X shape:", X.shape)
print("y shape:", y.shape)
print("qid shape:", qid.shape)
print("Number of unique queries:", len(np.unique(qid)))


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


X shape: (10425, 136)
y shape: (10425,)
qid shape: (10425,)
Number of unique queries: 100


## Data inspect

To get more insight into the data, we are inspecting:

- shape of X, number of unique queries
- label distribution as counts and percentages
- documents-per-query min, median, mean, max
- number of queries where every document has label 0
- min/max of the first 10 feature columns (to see the scale spread)
- whether there are NaNs

In [2]:
# --- Observational stats on the loaded chunk (X, y, qid) ---

# shape of X, number of unique queries
unique_qids, counts = np.unique(qid, return_counts=True)
print("X shape:", X.shape)
print("Number of unique queries:", len(unique_qids))

# label distribution: counts and percentages
labels, label_counts = np.unique(y, return_counts=True)
total = y.shape[0]
print("\nLabel distribution:")
for lab, cnt in zip(labels, label_counts):
    print(f"  label {int(lab)}: count={cnt}, pct={100 * cnt / total:.2f}%")

# documents per query: min, median, mean, max
print("\nDocuments per query:")
print("  min:   ", counts.min())
print("  median:", np.median(counts))
print("  mean:  ", counts.mean())
print("  max:   ", counts.max())

# number of queries where every document has label 0
all_zero_queries = 0
for q in unique_qids:
    mask = qid == q
    if np.all(y[mask] == 0):
        all_zero_queries += 1
print(f"\nQueries where every document has label 0: {all_zero_queries} / {len(unique_qids)}")

# min/max of the first 10 feature columns
print("\nFirst 10 feature columns -- min/max:")
for i in range(10):
    col = X[:, i]
    print(f"  feature {i+1}: min={col.min():.4f}, max={col.max():.4f}")

# NaNs
has_nan_X = np.isnan(X).any()
has_nan_y = np.isnan(y).any()
print(f"\nNaNs in X: {has_nan_X}")
print(f"NaNs in y: {has_nan_y}")



preview = pd.DataFrame(X[:5, :10], columns=[f"f{i+1}" for i in range(10)])
preview.insert(0, "label", y[:5])
preview.insert(0, "qid", qid[:5])
preview

X shape: (10425, 136)
Number of unique queries: 100

Label distribution:
  label 0: count=6087, pct=58.39%
  label 1: count=2861, pct=27.44%
  label 2: count=1279, pct=12.27%
  label 3: count=135, pct=1.29%
  label 4: count=63, pct=0.60%

Documents per query:
  min:    18
  median: 95.0
  mean:   104.25
  max:    337

Queries where every document has label 0: 3 / 100

First 10 feature columns -- min/max:
  feature 1: min=0.0000, max=7.0000
  feature 2: min=0.0000, max=6.0000
  feature 3: min=0.0000, max=6.0000
  feature 4: min=0.0000, max=6.0000
  feature 5: min=0.0000, max=7.0000
  feature 6: min=0.0000, max=1.0000
  feature 7: min=0.0000, max=1.0000
  feature 8: min=0.0000, max=1.0000
  feature 9: min=0.0000, max=1.0000
  feature 10: min=0.0000, max=1.0000

NaNs in X: False
NaNs in y: False


,qid,label,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10
0,1,2.0,3.0,3.0,0.0,0.0,3.0,1.0,1.0,0.000000,0.0,1.0
1,1,2.0,3.0,0.0,3.0,0.0,3.0,1.0,0.0,1.000000,0.0,1.0
2,1,0.0,3.0,0.0,2.0,0.0,3.0,1.0,0.0,0.666667,0.0,1.0
3,1,2.0,3.0,0.0,3.0,0.0,3.0,1.0,0.0,1.000000,0.0,1.0
4,1,1.0,3.0,0.0,3.0,0.0,3.0,1.0,0.0,1.000000,0.0,1.0


### Takeaway

- Label distribution is heavily skewed, with majority of lables being 0. This is normal for retrieval, as majority of retrieved documents are not relevant for given query. This also means that accuracy-based metrics should not be used - as predicting 0 every time would be evaluated decently, when it defintely should not.

- From doc-per-query stats, we see there are ~100 docs per query on average. This should be taken into account in case of a pairwise approach, as there can be a lot of possible pairs -> within one query, 100 docs give about (100x99)/2 ~ 5k possible doc pairs, so 100 queries could give up to 500k pairs (not all of them would be viable for pairwise, but still, it would need to be capped).

- There are ~3% queries, where every document has a label 0. When computing NDCG, these would have to be removed/ignored, as there would be division by IDCG=0. For a pointwise model, we can keep them in trainign, since learning to predict 0s is useful. A pairwise model skips them automatically, as they contain no pairs with different labels.

- First 10 features have different ranges: some go up to ~7, others only up to 1. This is why we scale the features later, so that no feature dominates just because of its range.

- There are no NaNs, so no need to discard invalid rows.

## Data splitting

We split the data into train and test sets so we can evaluate the model on data it has not seen. A plain random split of rows would not work here, because many rows share the same query_id. The model’s job is to predict document relevance for a new, unseen query, so the test set must contain only queries that were not used in training. If documents from the same query appeared in both sets, the model could benefit from having seen that query, which is data leakage and would inflate the evaluation results. So we split by query id: 80% of queries go to train, 20% to test.

In [3]:
# !pip install scikit-learn

from sklearn.model_selection import train_test_split

# Split at the QUERY level, not the row level -- this is essential for LTR data.
# Splitting rows randomly would let documents from the same query end up in
# both train and test, leaking information (the model could learn query-
# specific patterns from train that "help" on test just because it's the
# same query, not because it generalizes).

RANDOM_STATE = 42
TEST_SIZE = 0.20

unique_qids = np.unique(qid)

train_qids, test_qids = train_test_split(
    unique_qids,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

# Build boolean masks over the original flat arrays based on which split
# each row's query id falls into
train_mask = np.isin(qid, train_qids)
test_mask = np.isin(qid, test_qids)

X_train, y_train, qid_train = X[train_mask], y[train_mask], qid[train_mask]
X_test, y_test, qid_test = X[test_mask], y[test_mask], qid[test_mask]

print("Train:", X_train.shape, "| unique queries:", len(np.unique(qid_train)))
print("Test: ", X_test.shape, "| unique queries:", len(np.unique(qid_test)))

# --- Assertion: no shared query ids between train and test ---
train_qid_set = set(np.unique(qid_train))
test_qid_set = set(np.unique(qid_test))
overlap = train_qid_set & test_qid_set

assert len(overlap) == 0, f"Found {len(overlap)} overlapping query ids between train and test: {overlap}"
print("\nAssertion passed: no overlapping query ids between train and test.")

Train: (8255, 136) | unique queries: 80
Test:  (2170, 136) | unique queries: 20

Assertion passed: no overlapping query ids between train and test.


### Takeaway

Data was correctly into 80 trainind and 20 test queries. With no overlapping queries among the two.

Ratio between the two datasets' rows is 79/21, which is roughly 80/20.

## Data scaling

As model uses the same learning rate across all features, it is important to have them on the same scale (Ridge also benefits, as its penalty treats all weights equally). It is also important to scale them over train data only, and then to apply the same scale to the test set, as well as future inference. Test and inference samples can have values outside of the train scale, and this is okay, we will not clip these values, as that way we would lose information. Constant features are set to 0, as they provide no signal over multiple data and would cause division by 0 in the min-max scaling formula.

In [4]:
from sklearn.preprocessing import MinMaxScaler

# Fit min/max on TRAIN ONLY -- this avoids leaking test-set feature ranges
# into the scaling parameters. sklearn's MinMaxScaler already handles the
# "constant feature" case for us: internally, when max == min for a column,
# it substitutes a scale of 1 instead of dividing by zero, which yields
# (X - min) * 1 = 0 for every value in that column. So constant features
# come out as all-0, matching what you asked for, with no manual patching needed.

scaler = MinMaxScaler(clip=False)  # clip=False -> test values outside [0,1] stay outside
scaler.fit(X_train)

# transform() always returns a new array; it does not modify X_train/X_test in place
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape: ", X_test_scaled.shape)

# Sanity checks
print("\nTrain scaled min/max per column (first 20 cols):")
print("  min:", X_train_scaled[:, :20].min(axis=0))
print("  max:", X_train_scaled[:, :20].max(axis=0))

print("\nTest scaled min/max per column (first 20 cols) -- may fall outside [0,1], that's expected:")
print("  min:", X_test_scaled[:, :20].min(axis=0))
print("  max:", X_test_scaled[:, :20].max(axis=0))

# Confirm constant train-set columns collapsed to 0
constant_cols = np.where(scaler.data_min_ == scaler.data_max_)[0]
print(f"\nConstant feature columns in train set: {len(constant_cols)}")
if len(constant_cols) > 0:
    print("  Example constant col scaled values (should be all 0):",
          X_train_scaled[:5, constant_cols[0]])

X_train_scaled shape: (8255, 136)
X_test_scaled shape:  (2170, 136)

Train scaled min/max per column (first 20 cols):
  min: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  max: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]

Test scaled min/max per column (first 20 cols) -- may fall outside [0,1], that's expected:
  min: [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  3.33333333e-02  1.04004160e-04 -2.99748814e-02
  2.37191272e-02  2.29724412e-02  1.30025017e-01  1.17850249e-02]
  max: [0.85714286 1.5        1.2        1.5        0.85714286 1.
 1.         1.         1.         1.         1.01826241 0.02257965
 0.33166771 1.03333333 0.59812793 0.7935415  0.76546355 0.72527445
 0.85917944 0.80170942]

Constant feature columns in train set: 0


### Takeaway

We inspect first 20 features and see train ones scaled on [0-1], while as we mentioned, some test features are outside of this range, and that is okay.

Also, there are no constant features that would be annulated.

## Building evaluation

Before training the model, we build evaluation. Evaluation should match the goal of the project, so we should first analyze what the target output is.

The relevance labels are discrete values [0, 1, 2, 3, 4]. Even though they are discrete, they are not categorical (those would be classes such as A, B, C, with no relation to each other). These values are ordinal, as 4 > 3 > ... > 0. But they are also not truly continuous, as the gap between 3 and 4 might not be the same as between 1 and 2.

Therefore it makes sense to train a regression model, to produce values close to the relevance values. However, we should question what the end goal of this project is. If we ultimately need to predict the relevance labels, as we might then use them for something else, then it makes sense to train and evaluate the model on its precision in producing exact relevance labels. A straightforward evaluation method for this case is MSE (Mean Squared Error), where the model’s predictions are compared against the ground truth relevance labels.

However, if the end goal is to rank documents within a query based on relevance, then we do not necessarily care about the exact outputs of the model, we care more about the order in which the documents are ranked within a query. In that case, we can evaluate the model using NDCG (Normalized Discounted Cumulative Gain), where we take the model’s outputs, rank the documents based on them, compare that order to the ground truth one, and return a score that reflects ranking quality.

For the end goal of predicting exact relevance labels, we implement MSE as mean squared difference between model outputs and ground truth relevance labels, over the evaluated dataset. We also create couple unit tests, for perfect and constant imperfect scores.

In [5]:
def mean_mse(y_score, y_true):
    """
    Compute Mean Squared Error between predicted scores (treated as predicted
    relevance) and ground-truth relevance labels.

    No k, no query grouping -- flat comparison across all rows.
    """
    y_score = np.asarray(y_score, dtype=float)
    y_true = np.asarray(y_true, dtype=float)
    return np.mean((y_score - y_true) ** 2)

In [6]:
# --- Unit test 1: perfect predictions -> MSE == 0 ---
def test_mse_perfect():
    y_true = np.array([0, 1, 2, 3, 4])
    y_score = np.array([0, 1, 2, 3, 4])
    result = mean_mse(y_score, y_true)
    assert np.isclose(result, 0.0), f"Expected 0.0, got {result}"
    print("test_mse_perfect passed:", result)


# --- Unit test 2: constant diff of 2 -> MSE == 4 ---
def test_mse_constant_diff():
    y_true = np.array([0, 1, 2, 3, 4])
    y_score = y_true + 2  # every prediction is off by exactly 2
    result = mean_mse(y_score, y_true)
    assert np.isclose(result, 4.0), f"Expected 4.0, got {result}"
    print("test_mse_constant_diff passed:", result)


test_mse_perfect()
test_mse_constant_diff()

test_mse_perfect passed: 0.0
test_mse_constant_diff passed: 4.0


For the end goal of ranking the documents, we are implementing mean NDCG@k over the whole eval dataset. As these rankings only make sense within a query, we first have to compute NDCG@k for each given query, and then average the results over all queries. We will use exponential gain (2**relevance - 1) so that e.g. the gap between 3 and 4 is larger than gap between 1 and 2. and the standard log discount to penalize later ranked positions. DCG is calculated over the ground truth labels, in the order the model ranked the documents, and IDCG over the same labels in their ideal (descending) order. We use @k, as usually we only care about top k ranks (e.g. top 10).

We are creating 3 unit tests:
- perfect ordering that should return 1
- query with all 0s that should return NaN (as NDCG score can be a NaN, in case of all 0s)
- reversed perfect ordering that should return score well below 1

In [7]:
import numpy as np

def _dcg(sorted_relevances, k):
    """DCG using exponential gain (2^rel - 1) and log2 discount, over the
    first k relevances of an already-sorted (by score, descending) array."""
    sorted_relevances = np.asarray(sorted_relevances[:k], dtype=float)
    gains = 2 ** sorted_relevances - 1
    ranks = np.arange(1, len(sorted_relevances) + 1)  # i = 1, 2, 3, ... (1-indexed rank)
    discounts = 1.0 / np.log2(ranks + 1)  # matches 1/log2((i+1)+1) with i 0-indexed
    return np.sum(gains * discounts)


def ndcg_at_k(y_score, y_true, k):
    """
    Compute NDCG@k for a single query.

    y_score : model output scores, in original data order
    y_true  : ground-truth relevance labels, in the same order as y_score
    k       : cutoff rank

    Returns NaN if the ideal DCG (IDCG) is 0, i.e. every doc has relevance 0
    -- there's no meaningful ranking to evaluate against.
    """
    y_score = np.asarray(y_score, dtype=float)
    y_true = np.asarray(y_true, dtype=float)

    # Sort by predicted score, descending. Stable sort ensures ties break by
    # original order deterministically (not by index shuffling / undefined order).
    score_order = np.argsort(-y_score, kind="stable")
    dcg = _dcg(y_true[score_order], k)

    # Ideal ranking: sort ground-truth relevances themselves, descending, stable.
    ideal_order = np.argsort(-y_true, kind="stable")
    idcg = _dcg(y_true[ideal_order], k)

    if idcg == 0:
        return np.nan

    return dcg / idcg


def mean_ndcg(y_score, y_true, qids, k):
    """
    Compute mean NDCG@k across queries.

    y_score, y_true, qids : parallel arrays, one entry per (query, doc) row.
    k : cutoff rank

    Groups rows by query id, computes ndcg_at_k per query, drops any NaN
    results (queries with IDCG == 0), and returns (mean_ndcg, n_queries_used).
    """
    y_score = np.asarray(y_score, dtype=float)
    y_true = np.asarray(y_true, dtype=float)
    qids = np.asarray(qids)

    per_query_scores = []
    for q in np.unique(qids):
        mask = qids == q
        score = ndcg_at_k(y_score[mask], y_true[mask], k)
        per_query_scores.append(score)

    per_query_scores = np.array(per_query_scores, dtype=float)
    valid_scores = per_query_scores[~np.isnan(per_query_scores)]

    if len(valid_scores) == 0:
        return np.nan, 0

    return valid_scores.mean(), len(valid_scores)

In [8]:
# --- Unit test 1: perfect ordering -> NDCG should be 1 ---
def test_perfect_ordering():
    y_true = np.array([3, 2, 1, 0])
    y_score = np.array([4.0, 3.0, 2.0, 1.0])  # already in the same order as y_true, descending
    result = ndcg_at_k(y_score, y_true, k=4)
    assert np.isclose(result, 1.0), f"Expected 1.0, got {result}"
    print("test_perfect_ordering passed:", result)


# --- Unit test 2: query with all-0 relevance -> NaN ---
def test_all_zero_relevance():
    y_true = np.array([0, 0, 0, 0])
    y_score = np.array([0.9, 0.1, 0.5, 0.3])
    result = ndcg_at_k(y_score, y_true, k=4)
    assert np.isnan(result), f"Expected NaN, got {result}"
    print("test_all_zero_relevance passed:", result)


# --- Unit test 3: reversed perfect ordering -> score clearly below 1 ---
def test_reversed_ordering():
    y_true = np.array([3, 2, 1, 0])
    y_score = np.array([1.0, 2.0, 3.0, 4.0])  # exactly reversed vs. y_true's order
    result = ndcg_at_k(y_score, y_true, k=4)
    assert result < 0.95, f"Expected clearly below 1.0, got {result}"
    print("test_reversed_ordering passed:", result)


test_perfect_ordering()
test_all_zero_relevance()
test_reversed_ordering()

test_perfect_ordering passed: 1.0
test_all_zero_relevance passed: nan
test_reversed_ordering passed: 0.547831481922746


### Takeaway

The unit tests return expected values for both MSE and NDCG evaluations. 

We add one more test that compares the two metrics. If the model outputs are ranked perfectly but are off from the ground truth by a constant (e.g. every score is label + 2), MSE reports poor performance (4.0), while NDCG correctly evaluates the ranking as perfect (1.0). This shows that if we care about ranking, MSE alone can be misleading.

In [9]:
def test_constant_diff_perfect_ordering():
    y_true = np.array([0, 1, 2, 3, 4])
    y_score = y_true + 2  # every prediction is off by exactly 2
    result = ndcg_at_k(y_score, y_true, k=5)
    assert np.isclose(result, 1.0), f"Expected 1.0, got {result}"
    print("test_perfect_ordering passed:", result)

test_constant_diff_perfect_ordering()
test_mse_constant_diff()

test_perfect_ordering passed: 1.0
test_mse_constant_diff passed: 4.0


## Baselines

Before training the model we should implement the baselines and get their evaluation results. Without them, we can't tell how much performance is gained by training a model.

We will create baselines for both NDCG as well as MSE evaluations.

For MSE, we could simply assign random scores in range [0, 4], however for MSE, that is not a pure baseline, as assigning random scores can make this worse than "no learning" floor baseline. The mathematical floor baseline for MSE would be to have all predictions equal to the mean of all labels, as that minimizes squared error. If we were to remove all features, the linear model would colapse to predicting the mean value as bias constant, to minimize the MSE. 

It is also important to consider values only within the train set, for the mean value, and then to evaluate this baseline on the test set, so that there is no data leakage.

In [10]:
# ============================================================
# Naive baseline: predict the mean (constant prediction) for every row
# ============================================================
# Mean computed from TRAIN labels only -- consistent with the "fit on train,
# evaluate on test" discipline we've used for scaling and the model. This is
# the standard naive baseline: it uses no features, just a constant.

mean_label = y_train.mean()
print(f"Predicted constant (mean of train labels): {mean_label:.4f}")

# Apply the same constant prediction to every test row
y_pred_mean_baseline = np.full(y_test.shape[0], mean_label)

mean_baseline_mse = mean_mse(y_pred_mean_baseline, y_test)

print("\n=== Predict-the-mean baseline ===")
print(f"  Test MSE: {mean_baseline_mse:.4f}")

Predicted constant (mean of train labels): 0.5695

=== Predict-the-mean baseline ===
  Test MSE: 0.6510


For NDCG ranking evaluation, we will implement two baselines:
- Random: we assign random scores (range is irrelevant for ndcg, as we will rank them).
- Best-single-feature: we iterate over the features and treat each of them as scores, then we rank and evaluate ndcg@k. The feature with best ranking score will be used as the baseline. The idea here is to find baseline performance that is already stored in the data (although naivly, in a single feature), without any training. Again, it is important to consider only features from the train set, but not only that, during search for the best feature, we must also evaluate ndcg@k over the train set, due to data leakage. Only after we found the best ranking feature, then we evalaute on the test set as final baseline result.

In [11]:
K = 10
# ============================================================
# Baseline 1: Random -- uniform random scores, fixed seed
# ============================================================
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
random_scores = rng.uniform(0, 1, size=y_test.shape[0])

random_ndcg, random_n_queries = mean_ndcg(random_scores, y_test, qid_test, k=K)

print("\n=== Random baseline ===")
print(f"  Mean NDCG@{K}: {random_ndcg:.4f}")
print(f"  Test queries used: {random_n_queries} / {len(np.unique(qid_test))}")


# ============================================================
# Baseline 2: Best single feature
# ============================================================
# Selection happens on TRAIN only: try each of the 136 features as a
# standalone "score", measure its NDCG@10 on train, and pick the feature
# index with the highest train NDCG. We only use the *index* from this --
# the actual reported baseline score comes from applying that same feature
# on the TEST set, so selection and evaluation stay properly separated.

n_features = X_train.shape[1]
train_feature_ndcgs = np.full(n_features, np.nan)

for feat_idx in range(n_features):
    feat_scores = X_train[:, feat_idx]
    ndcg_val, _ = mean_ndcg(feat_scores, y_train, qid_train, k=K)
    train_feature_ndcgs[feat_idx] = ndcg_val

best_feat_idx = np.nanargmax(train_feature_ndcgs)
best_feat_train_ndcg = train_feature_ndcgs[best_feat_idx]

print(f"\nBest single feature selected on TRAIN: feature index {best_feat_idx} "
      f"(0-indexed, i.e. f{best_feat_idx + 1}), train NDCG@{K} = {best_feat_train_ndcg:.4f}")

# Now evaluate that same feature's column as the score on the TEST set
best_feat_test_scores = X_test[:, best_feat_idx]
best_feat_ndcg, best_feat_n_queries = mean_ndcg(best_feat_test_scores, y_test, qid_test, k=K)

print("\n=== Best single feature baseline ===")
print(f"  Feature used: f{best_feat_idx + 1}")
print(f"  Mean NDCG@{K}: {best_feat_ndcg:.4f}")
print(f"  Test queries used: {best_feat_n_queries} / {len(np.unique(qid_test))}")


# ============================================================
# Summary
# ============================================================
print("\n=== Summary (test set) ===")
print(f"  Random:               NDCG@{K} = {random_ndcg:.4f}  (queries used: {random_n_queries})")
print(f"  Best single feature:  NDCG@{K} = {best_feat_ndcg:.4f}  (queries used: {best_feat_n_queries}, f{best_feat_idx + 1})")


=== Random baseline ===
  Mean NDCG@10: 0.2001
  Test queries used: 19 / 20

Best single feature selected on TRAIN: feature index 107 (0-indexed, i.e. f108), train NDCG@10 = 0.3390

=== Best single feature baseline ===
  Feature used: f108
  Mean NDCG@10: 0.3566
  Test queries used: 19 / 20

=== Summary (test set) ===
  Random:               NDCG@10 = 0.2001  (queries used: 19)
  Best single feature:  NDCG@10 = 0.3566  (queries used: 19, f108)


### Takeaway

MSE baseline scores 0.651. This is the best we can do without using any features: always predicting the mean of the train labels.

NDCG baseline with random score ranking scores 0.2.

For the best-single-feature baseline, the best performing feature was f108, with 0.3390 NDCG@10 over the train set. On the test set it performs even better, with 0.3566. We might expect the feature to do worse on test, since it was selected on train. But the train and test scores are averages over different queries, so a small gap in either direction can happen by chance, especially with only 19 test queries.

Now that we have the baselines, we have reference points to judge the models against: a model needs to beat these to be worth using.

## Model choice

To decide on the model, we take into consideration the type of the data (features) as well as the output target (label). The features are numerical, which alone does not narrow the model choice significantly (e.g. linear, trees, neural nets, etc.). The output is one score per document, which is either a predicted relevance label or a ranking score, depending on the approach we choose (as mentioned, the goal being exact relevance labels or ranking).

As this exercise is more of an explanatory one, we will use linear models. They are simple, easy to interpret, and the same linear scoring function can serve both goals, so we can compare the two approaches directly. The trade-off is that linear models cannot capture feature interactions, so we do not expect state-of-the-art results, and the focus is on comparing approaches.

say that each model,eval option is possible but does not make sense

### 1. Pointwise (Linear Regression)

A linear regression model is a straightforward fit for this type of task: numerical features are fed into the model, and its weights are trained to produce a relevance score close to the ground truth label. This is the pointwise approach, where each document is scored independently of the others as a single point. For a document with feature vector x, the prediction is ŷ = w · x + b, a weighted sum of the features plus a bias. The weights are fit by minimizing the squared error between ŷ and the ground true label, which is why this approach fits the first goal best: predicting exact relevance labels.

Regarding evaluation, MSE is the natural metric, as it matches the training objective. However, we can also use NDCG, by ranking the documents within each query by their predicted scores. This is a relevant metric on its own, especially for comparing against the second approach. We use the raw predictions, without rounding or clipping, so that ties are rare and the ordering is not distorted.

For the implementation, we will use LinearRegression from sklearn. Fit it over the scaled train set and evaluate on scaled test set. Then compare the results with both MSE and NDCG baselines.

In [12]:
from sklearn.linear_model import LinearRegression

K = 10
results = {}  # name -> test metrics

def fit_and_evaluate(name, model):
    """Train on scaled train features vs. raw labels, evaluate on test."""
    model.fit(X_train_scaled, y_train)

    # Raw predictions: no rounding, no clipping
    scores_test = model.predict(X_test_scaled)

    ndcg, n_queries = mean_ndcg(scores_test, y_test, qid_test, k=K)
    mse = mean_mse(scores_test, y_test)

    print(f"=== {name} ===")
    print(f"  Mean NDCG@{K}: {ndcg:.4f}  (test queries used: {n_queries} / {len(np.unique(qid_test))})")
    print(f"  Test MSE:     {mse:.4f}\n")

    results[name] = {"ndcg": ndcg, "mse": mse}
    return model

def print_comparison():
    print("=== Comparison (test set) ===")
    print(f"{'Method':<28}{'NDCG@' + str(K):>10}{'MSE':>10}")
    print(f"{'Random uniform':<28}{random_ndcg:>10.4f}{'-':>10}")
    print(f"{'Best single feature':<28}{best_feat_ndcg:>10.4f}{'-':>10}")
    print(f"{'Train-mean constant':<28}{'-':>10}{mean_baseline_mse:>10.4f}")
    for name, r in results.items():
        print(f"{name:<28}{r['ndcg']:>10.4f}{r['mse']:>10.4f}")

lin_reg = fit_and_evaluate("Linear Regression", LinearRegression())
print_comparison()

=== Linear Regression ===
  Mean NDCG@10: 0.4048  (test queries used: 19 / 20)
  Test MSE:     1.0430

=== Comparison (test set) ===
Method                         NDCG@10       MSE
Random uniform                  0.2001         -
Best single feature             0.3566         -
Train-mean constant                  -    0.6510
Linear Regression               0.4048    1.0430


### Takeaway

The results show an improvement of ~0.05 NDCG@10 points over the best single feature. However, MSE is much worse than the predict-the-mean baseline.

This is a red flag, especially since MSE is the direct objective of the pointwise training method. We need to analyze why this happened and try to improve it. To understand whether this is overfitting, underfitting, or something else, we run the same evaluation on the train set. This is not something we use to assess the quality of the model, but it is useful for diagnosing poor test performance. If the model performs much better on train than on test, it is a case of overfitting. If it performs poorly on train as well, it is underfitting, which means we might have an issue with the data, model, or training choice.

In [13]:
train_results = {}  # name -> train metrics

# --- Train-side baselines (same definitions as on test) ---
# Random: same seed, uniform scores over the train rows
rng_train = np.random.default_rng(RANDOM_SEED)
random_scores_train = rng_train.uniform(0, 1, size=y_train.shape[0])
random_ndcg_train, _ = mean_ndcg(random_scores_train, y_train, qid_train, k=K)

# Best single feature: already selected on train (best_feat_idx), score is best_feat_train_ndcg
# Train-mean constant: the same constant, evaluated on train labels
mean_baseline_train_mse = mean_mse(np.full(y_train.shape[0], y_train.mean()), y_train)

def evaluate_on_train(name, model):
    """Evaluate an already-fitted model on the train set."""
    scores_train = model.predict(X_train_scaled)
    ndcg, n_queries = mean_ndcg(scores_train, y_train, qid_train, k=K)
    mse = mean_mse(scores_train, y_train)

    print(f"=== {name} (TRAIN) ===")
    print(f"  Mean NDCG@{K}: {ndcg:.4f}  (train queries used: {n_queries} / {len(np.unique(qid_train))})")
    print(f"  Train MSE:    {mse:.4f}\n")

    train_results[name] = {"ndcg": ndcg, "mse": mse}

def print_train_vs_test():
    print("=== Train vs. test ===")
    print(f"{'Method':<28}{'Train NDCG@' + str(K):>15}{'Test NDCG@' + str(K):>15}{'Train MSE':>12}{'Test MSE':>12}")
    print(f"{'Random uniform':<28}{random_ndcg_train:>15.4f}{random_ndcg:>15.4f}{'-':>12}{'-':>12}")
    print(f"{'Best single feature':<28}{best_feat_train_ndcg:>15.4f}{best_feat_ndcg:>15.4f}{'-':>12}{'-':>12}")
    print(f"{'Train-mean constant':<28}{'-':>15}{'-':>15}{mean_baseline_train_mse:>12.4f}{mean_baseline_mse:>12.4f}")
    for name in train_results:
        tr, te = train_results[name], results[name]
        print(f"{name:<28}{tr['ndcg']:>15.4f}{te['ndcg']:>15.4f}{tr['mse']:>12.4f}{te['mse']:>12.4f}")

evaluate_on_train("Linear Regression", lin_reg)
print_train_vs_test()

=== Linear Regression (TRAIN) ===
  Mean NDCG@10: 0.4516  (train queries used: 78 / 80)
  Train MSE:    0.5092

=== Train vs. test ===
Method                        Train NDCG@10   Test NDCG@10   Train MSE    Test MSE
Random uniform                       0.1791         0.2001           -           -
Best single feature                  0.3390         0.3566           -           -
Train-mean constant                       -              -      0.6357      0.6510
Linear Regression                    0.4516         0.4048      0.5092      1.0430


### Takeaway

The train MSE is half of the test MSE, and on train the model beats the train-mean baseline (0.509 vs 0.636), while on test it is worse than the baseline (1.043 vs 0.651). This is a classic case of overfitting: the model fits the train data well but does not generalize on test data.

Lack of data can be a reason for overfitting. In terms of rows and features we seem fine (~8k rows, 136 features), but the rows come from only 80 queries, and rows within a query are similar to each other, so the effective amount of data is smaller than the row count suggests. More queries would likely help.

To deal with overfitting, we can add regularization. To keep the same linear model, we use Ridge regression, which adds an L2 penalty on the squared weights to the training loss. This discourages large weights, so the model cannot fit noise as easily. The strength is set by alpha (alpha = 0 is plain linear regression, larger means stronger regularization). Ridge penalizes all weights equally, which is another reason we scaled the features.

In [14]:
from sklearn.linear_model import Ridge

ridge = fit_and_evaluate("Ridge", Ridge(alpha=1.0))
evaluate_on_train("Ridge", ridge)

print_comparison()
print_train_vs_test()

=== Ridge ===
  Mean NDCG@10: 0.4280  (test queries used: 19 / 20)
  Test MSE:     0.5613

=== Ridge (TRAIN) ===
  Mean NDCG@10: 0.4376  (train queries used: 78 / 80)
  Train MSE:    0.5205

=== Comparison (test set) ===
Method                         NDCG@10       MSE
Random uniform                  0.2001         -
Best single feature             0.3566         -
Train-mean constant                  -    0.6510
Linear Regression               0.4048    1.0430
Ridge                           0.4280    0.5613
=== Train vs. test ===
Method                        Train NDCG@10   Test NDCG@10   Train MSE    Test MSE
Random uniform                       0.1791         0.2001           -           -
Best single feature                  0.3390         0.3566           -           -
Train-mean constant                       -              -      0.6357      0.6510
Linear Regression                    0.4516         0.4048      0.5092      1.0430
Ridge                                0.4376    

### Takeaway

Ridge reduces the test MSE from 1.043 to 0.561, and now beats the predict-the-mean baseline (0.651). Test NDCG@10 also improves (0.405 to 0.428). The train metrics get slightly worse, as expected from regularization: the model fits the train data a bit less closely, but generalizes much better. The gap between train and test MSE is now small, so the overfitting is largely resolved. Alpha was left at the default (1.0) and not tuned.

Pointwise treats every row as an independent example, ignoring which query it belongs to. But relevance is judged within a query: the same feature values can correspond to different labels in different queries (e.g. a high bm25 may be ordinary for one query and exceptional for another). The model can only average over this. Also, minimizing label error for each document does not directly optimize the order of documents within a query, which is what we care about for ranking. Pairwise approaches address both by training only on comparisons between documents of the same query.

## 2. Pairwise (Logistic Regression)

As explained, the pointwise approach treats each row as a separate point, even though rows that share a query should be treated together. The pairwise approach addresses this by comparing only documents within the same query. It also changes the training objective: instead of predicting the exact relevance label, the model learns to score documents relative to each other, by relevance to the query.

It is called pairwise because it processes documents in pairs. Within each query, we create pairs of documents with different relevance labels, and train the model to score the more relevant one higher. The model keeps the same linear scoring function, score = w · x (no bias, explained below). The difference is in the training loop: instead of comparing the score to a target label, as linear regression does, we take the two scores of a pair and apply a sigmoid to their difference. If document A is more relevant than document B, we want sigmoid(score(A) − score(B)) to be close to 1. If the model scores them equally, sigmoid(0) = 0.5, so the loss is high, and the weights are updated to push the scores apart in the right direction. (This is equivalent to logistic regression on the feature difference x_A − x_B. The approach is known as RankNet).

One limitation is that every pair has the same target of 1, whether the labels differ by 1 (4 vs 3) or by 4 (4 vs 0). LambdaRank improves on this by weighting each pair by how much NDCG would change if the two documents swapped places, so it focuses on the pairs that matter most for the ranking metric. We do not implement it here.

We implement this in plain numpy, as sklearn has no built-in pairwise ranking model. For each query, we sample up to 100 pairs with different labels (to keep training fast). For each pair, we compute the difference of the two linear scores and apply the sigmoid. The error is sigmoid − 1, and we update the weights with a plain SGD step: w = w − lr · error · (x_i − x_j). During training we track and print the average loss per epoch, using the log loss L = −log(sigmoid). There is no bias term, as it cancels out in the difference of the two scores.

For evaluation, NDCG is the natural metric, as the training objective is ranking and not predicting exact labels. MSE makes no sense here: the scores only encode order and have no fixed scale (shifting or stretching them does not change the ranking), so comparing them to the 0-4 labels is meaningless. We still report it later to show this.

In [15]:
def get_query_pairs(labels, max_pairs, rng):
    """
    Build (i, j) index pairs *within* a single query's documents where
    label[i] > label[j] -- i.e. i should be ranked above j.

    labels    : relevance labels for this query's documents (local indices)
    max_pairs : cap on how many pairs to sample for this query
    rng       : np.random.Generator, for reproducible sampling

    Returns an array of shape (n_pairs, 2) of local index pairs.
    """
    n = len(labels)
    pairs = []
    for i in range(n):
        for j in range(n):
            if labels[i] > labels[j]:
                pairs.append((i, j))

    if len(pairs) == 0:
        return np.empty((0, 2), dtype=int)

    pairs = np.array(pairs, dtype=int)

    if len(pairs) > max_pairs:
        chosen_idx = rng.choice(len(pairs), size=max_pairs, replace=False)
        pairs = pairs[chosen_idx]

    return pairs


def train_pairwise(X, y, qid, n_epochs=5, lr=0.01, max_pairs_per_query=100, seed=42):
    """
    Train a linear pairwise (RankNet-style) scorer with plain SGD
    (one pair = one gradient step).

    X   : (n_samples, n_features) feature matrix
    y   : (n_samples,) relevance labels
    qid : (n_samples,) query ids, parallel to X/y

    No bias term -- it cancels out in (x_i - x_j), so we don't model or learn one.

    Returns the learned weight vector w, shape (n_features,).
    """
    rng = np.random.default_rng(seed)
    n_features = X.shape[1]
    w = np.zeros(n_features)

    unique_qids = np.unique(qid)

    for epoch in range(n_epochs):
        epoch_qids = rng.permutation(unique_qids)

        total_pairs_used = 0
        total_loss = 0.0

        for q in epoch_qids:
            mask = qid == q
            X_q = X[mask]
            y_q = y[mask]

            if len(y_q) < 2:
                continue

            pairs = get_query_pairs(y_q, max_pairs_per_query, rng)

            for i, j in pairs:
                x_diff = X_q[i] - X_q[j]
                score_diff = w @ x_diff
                sigmoid = 1.0 / (1.0 + np.exp(-score_diff))
                target = 1.0
                error = sigmoid - target

                w = w - lr * error * x_diff

                total_pairs_used += 1
                total_loss += -np.log(sigmoid + 1e-12)

        avg_loss = total_loss / total_pairs_used if total_pairs_used > 0 else float("nan")
        print(f"Epoch {epoch + 1}/{n_epochs} -- pairs used: {total_pairs_used}, avg loss: {avg_loss:.4f}")

    return w

In [16]:
class PairwiseLinearRanker:
    """Thin wrapper so the numpy RankNet can be used like a sklearn model."""

    def __init__(self, qid_train, **train_kwargs):
        self.qid_train = qid_train        # query ids of the training rows
        self.train_kwargs = train_kwargs  # n_epochs, lr, max_pairs_per_query, seed

    def fit(self, X, y):
        self.w_ = train_pairwise(X, y, self.qid_train, **self.train_kwargs)
        return self

    def predict(self, X):
        # linear scorer, no bias term
        return X @ self.w_


pairwise = fit_and_evaluate(
    "Pairwise (RankNet)",
    PairwiseLinearRanker(qid_train, n_epochs=20, lr=0.01, max_pairs_per_query=100, seed=42),
)
evaluate_on_train("Pairwise (RankNet)", pairwise)

print_train_vs_test()

Epoch 1/20 -- pairs used: 7663, avg loss: 0.6154
Epoch 2/20 -- pairs used: 7663, avg loss: 0.5925
Epoch 3/20 -- pairs used: 7663, avg loss: 0.5967
Epoch 4/20 -- pairs used: 7663, avg loss: 0.5896
Epoch 5/20 -- pairs used: 7663, avg loss: 0.5908
Epoch 6/20 -- pairs used: 7663, avg loss: 0.5885
Epoch 7/20 -- pairs used: 7663, avg loss: 0.5917
Epoch 8/20 -- pairs used: 7663, avg loss: 0.5886
Epoch 9/20 -- pairs used: 7663, avg loss: 0.5825
Epoch 10/20 -- pairs used: 7663, avg loss: 0.5859
Epoch 11/20 -- pairs used: 7663, avg loss: 0.5915
Epoch 12/20 -- pairs used: 7663, avg loss: 0.5844
Epoch 13/20 -- pairs used: 7663, avg loss: 0.5900
Epoch 14/20 -- pairs used: 7663, avg loss: 0.5785
Epoch 15/20 -- pairs used: 7663, avg loss: 0.5846
Epoch 16/20 -- pairs used: 7663, avg loss: 0.5815
Epoch 17/20 -- pairs used: 7663, avg loss: 0.5853
Epoch 18/20 -- pairs used: 7663, avg loss: 0.5859
Epoch 19/20 -- pairs used: 7663, avg loss: 0.5882
Epoch 20/20 -- pairs used: 7663, avg loss: 0.5786
=== Pairw

Training loss: 

- With zero-initialized weights every pair starts at sigmoid(0) = 0.5, so the loss starts at log(2) ≈ 0.693 (the “no learning” level). The epoch-1 average (0.615) is already lower, since the weights update during the epoch. The loss then drops quickly and plateaus around 0.58 to 0.59, with small epoch-to-epoch noise coming from the random pair sampling and single-pair SGD steps. So the model learns something real (clearly below 0.693), but stops improving early.

Ranking quality:

- Test NDCG@10 is 0.425, well above the best single feature (0.357) and random (0.200), and on par with Ridge (0.428). So pairwise training is not clearly better than pointwise here. With only 19 test queries, a difference this small is within noise. Train and test NDCG are close (0.422 vs 0.425), so there is no overfitting; if anything the model is limited in what it can fit.

MSE:
- The pairwise MSE (~9) is meaningless, as expected: the scores are only meant to order documents, and are not on the 0 to 4 label scale. This is the case where NDCG is the right metric and MSE is misleading.

The plateau raises a question: is it the optimizer (plain SGD) or the linear model itself? We check this next.

### Sanity check

Before investigating the plateau, we check that the update rule itself is correct. We take a single pair (label 4 vs label 0) from one query and run many gradient steps on that pair alone. If the implementation is right, sigmoid(score difference) should climb toward 1 and the loss should drop toward 0. If it does not, there is a bug, not a data or tuning issue.

In [17]:
# ============================================================
# Sanity check: isolate ONE clean pair (label 4 vs label 0) from ONE query,
# and run many gradient steps on it alone. If the pairwise update logic is
# correct, sigmoid(score_diff) should climb toward 1 and loss should drop
# toward 0 -- if it doesn't, that points to a real bug, not a data/tuning issue.
# ============================================================

# --- Find a query in train that has both a label-4 doc and a label-0 doc ---
found_qid = None
i_local, j_local = None, None

for q in np.unique(qid_train):
    mask = qid_train == q
    y_q = y_train[mask]
    if (y_q == 4).any() and (y_q == 0).any():
        found_qid = q
        i_local = np.where(y_q == 4)[0][0]  # first label-4 doc
        j_local = np.where(y_q == 0)[0][0]  # first label-0 doc
        X_q = X_train_scaled[mask]
        break

assert found_qid is not None, "No query with both label 4 and label 0 found in train set"

x_i = X_q[i_local]
x_j = X_q[j_local]
x_diff = x_i - x_j

print(f"Using query id {found_qid}: doc with label 4 vs doc with label 0")
print(f"||x_diff|| = {np.linalg.norm(x_diff):.4f}  (0 would mean identical features -- can't separate)")

# --- Isolated training loop: fresh weights, only this one pair, many steps ---
w_check = np.zeros(X_train_scaled.shape[1])
lr = 0.05
n_steps = 500

sigmoids = []
losses = []

for step in range(n_steps):
    score_diff = w_check @ x_diff
    sigmoid = 1.0 / (1.0 + np.exp(-score_diff))
    target = 1.0
    error = sigmoid - target

    w_check = w_check - lr * error * x_diff

    sigmoids.append(sigmoid)
    losses.append(-np.log(sigmoid + 1e-12))

print(f"\nStep 0:    sigmoid = {sigmoids[0]:.6f}, loss = {losses[0]:.6f}")
print(f"Step {n_steps // 2}:  sigmoid = {sigmoids[n_steps // 2]:.6f}, loss = {losses[n_steps // 2]:.6f}")
print(f"Step {n_steps - 1}: sigmoid = {sigmoids[-1]:.6f}, loss = {losses[-1]:.6f}")

assert sigmoids[-1] > sigmoids[0], "Sigmoid did not increase -- likely a bug in the update"
assert losses[-1] < losses[0], "Loss did not decrease -- likely a bug in the update"
assert sigmoids[-1] > 0.9, f"Expected sigmoid to climb well above 0.5, got {sigmoids[-1]:.4f}"

print("\nSanity check passed: sigmoid climbed toward 1, loss dropped toward 0, on an isolated clean pair.")

Using query id 7: doc with label 4 vs doc with label 0
||x_diff|| = 1.0890  (0 would mean identical features -- can't separate)

Step 0:    sigmoid = 0.500000, loss = 0.693147
Step 250:  sigmoid = 0.930048, loss = 0.072519
Step 499: sigmoid = 0.964738, loss = 0.035898

Sanity check passed: sigmoid climbed toward 1, loss dropped toward 0, on an isolated clean pair.


### Takeaway

The sigmoid climbs from 0.5 to about 0.96 after 500 steps and the loss drops from 0.693 to about 0.036, so the update direction is correct. It does not reach exactly 1 because the logistic loss flattens as the sigmoid approaches 1: the gradient (sigmoid − 1) shrinks, so progress slows. This check uses a single pair, so it does not rule out every possible bug, but together with the loss dropping below log(2) on the real data, it gives us confidence in the implementation.

### Why does the loss stop improving?

The loss stalls around 0.58 to 0.59. There are two possible reasons: either better weights exist and plain SGD (constant learning rate, noisy single-pair steps) fails to find them, or a linear scorer simply cannot get lower than this on our data. To tell the two apart, we solve the same objective exactly: logistic regression on the pair differences (x_i − x_j), with each pair included in both directions (difference with label 1, negated difference with label 0) and no intercept. This has no SGD noise. We then compare the pair log-loss of both solutions on the same fixed set of train pairs. If the exact solution reaches a clearly lower loss, SGD was the limit. If the losses are about equal, the linear model is the limit.

In [18]:
from sklearn.linear_model import LogisticRegression

def build_pair_diffs(X, y, qid, max_pairs_per_query, rng):
    """For each query, sample up to max_pairs_per_query pairs with label_i > label_j
    and return the matrix of feature differences (x_i - x_j)."""
    diffs = []
    for q in np.unique(qid):
        idx = np.where(qid == q)[0]
        y_q = y[idx]
        i, j = np.where(y_q[:, None] > y_q[None, :])
        if len(i) == 0:
            continue
        if len(i) > max_pairs_per_query:
            keep = rng.choice(len(i), size=max_pairs_per_query, replace=False)
            i, j = i[keep], j[keep]
        diffs.append(X[idx[i]] - X[idx[j]])
    return np.vstack(diffs)

def pair_logloss(w, D):
    """Mean -log(sigmoid(w . d)) over pair differences D (numerically stable)."""
    return np.mean(np.logaddexp(0.0, -(D @ w)))


class PairwiseLogReg:
    """Exact solve of the pairwise objective: logistic regression on pair differences,
    both directions (d -> 1, -d -> 0), no intercept."""
    def __init__(self, qid_train, max_pairs_per_query=200, C=10.0, seed=42):
        self.qid_train = qid_train
        self.max_pairs = max_pairs_per_query
        self.C = C
        self.seed = seed

    def fit(self, X, y):
        rng = np.random.default_rng(self.seed)
        D = build_pair_diffs(X, y, self.qid_train, self.max_pairs, rng)
        X_pairs = np.vstack([D, -D])
        y_pairs = np.concatenate([np.ones(len(D)), np.zeros(len(D))])
        clf = LogisticRegression(fit_intercept=False, C=self.C, max_iter=2000)
        clf.fit(X_pairs, y_pairs)
        self.w_ = clf.coef_.ravel()
        return self

    def predict(self, X):
        return X @ self.w_


# --- Fit the exact solution and add it to the tables ---
pairwise_exact = fit_and_evaluate("Pairwise exact (LogReg)", PairwiseLogReg(qid_train))
evaluate_on_train("Pairwise exact (LogReg)", pairwise_exact)

# --- Compare pair log-loss on the same fixed set of train pairs ---
D_eval = build_pair_diffs(X_train_scaled, y_train, qid_train, 200, np.random.default_rng(0))
print("\n=== Pair log-loss on fixed train pairs (lower is better) ===")
print(f"  Zero weights (no learning):  {pair_logloss(np.zeros(X_train_scaled.shape[1]), D_eval):.4f}")
print(f"  RankNet (our SGD):           {pair_logloss(pairwise.w_, D_eval):.4f}")
print(f"  Exact LogReg:                {pair_logloss(pairwise_exact.w_, D_eval):.4f}")

print_comparison()
print_train_vs_test()

=== Pairwise exact (LogReg) ===
  Mean NDCG@10: 0.4501  (test queries used: 19 / 20)
  Test MSE:     1.7030

=== Pairwise exact (LogReg) (TRAIN) ===
  Mean NDCG@10: 0.4437  (train queries used: 78 / 80)
  Train MSE:    1.1179


=== Pair log-loss on fixed train pairs (lower is better) ===
  Zero weights (no learning):  0.6931
  RankNet (our SGD):           0.5989
  Exact LogReg:                0.5691
=== Comparison (test set) ===
Method                         NDCG@10       MSE
Random uniform                  0.2001         -
Best single feature             0.3566         -
Train-mean constant                  -    0.6510
Linear Regression               0.4048    1.0430
Ridge                           0.4280    0.5613
Pairwise (RankNet)              0.4247    8.9942
Pairwise exact (LogReg)         0.4501    1.7030
=== Train vs. test ===
Method                        Train NDCG@10   Test NDCG@10   Train MSE    Test MSE
Random uniform                       0.1791         0.2001           

### Takeaway

The exact solution reaches a lower pair log-loss (0.569) than our SGD (0.599), so our manual optimizer was not optimal: constant learning rate and single-pair steps leave it short of the best linear weights. The exact solver also has the highest test NDCG@10 (0.450), but with 19 test queries this difference is within noise, so we do not claim it ranks better. We did not tune the SGD (e.g. learning rate decay, mini-batches, more epochs), which would likely close part of the gap, and we leave that as a possible next step. The takeaway is that RankNet with a linear scorer is logistic regression on pair differences, and when a linear scorer is enough, solving it exactly is the more reliable route.

## Conclusion

What we did:
- We built a small learning-to-rank pipeline on a 100-query sample of MSLR-WEB10k: a query-level train/test split, min-max scaling fit on train only, NDCG@10 implemented from scratch, and baselines to judge every model against.

Baselines:
- Random scoring reaches an NDCG@10 of 0.200, the best single feature (f108) reaches 0.357, and always predicting the train mean gives an MSE of 0.651.

Pointwise:
- Plain linear regression beat the baselines on NDCG (0.405) but was worse than the predict-the-mean baseline on MSE (1.043 vs 0.651). Train results (MSE 0.509) showed this was overfitting. Ridge regularization fixed most of it: test MSE dropped to 0.561 and NDCG@10 rose to 0.428, with alpha left untuned.

Pairwise:
- A linear RankNet implemented in numpy reached a test NDCG@10 of 0.425, on par with Ridge, not clearly better. Its MSE (~9) is meaningless, because ranking scores have no fixed scale, which is why NDCG is the right metric here. A one-pair sanity check confirmed that the update rule works.

Optimizer check:
- Solving the same objective exactly with logistic regression on pair differences gave a lower pair log-loss (0.569 vs 0.599) and the highest test NDCG@10 (0.450). So our plain SGD was not optimal. It also shows that a linear RankNet is logistic regression on pair differences.

Main takeaways:

- Baselines are essential. They showed that plain linear regression looked fine on NDCG but failed on MSE.
- Comparing train and test results is what diagnosed the overfitting.
- The metric must match the goal: MSE for exact labels, NDCG for ranking.
- With only 19 test queries, NDCG differences of 0.02 to 0.03 between the linear models are within noise, so we do not claim that one approach ranks better.

Limitations:

- Small sample (100 queries) and a single train/test split.
- Linear models only, so no feature interactions.
- Hyperparameters (alpha, epochs, learning rate, pairs per query) were not tuned on a validation set.
- Min-max scaling without clipping lets out-of-range test values pass straight through to linear models.

What can be done next:

- Use more queries and average over several query-level splits or cross-validation, to get reliable comparisons.
- Tune hyperparameters on a validation split of the train queries.
- Improve the SGD (learning rate decay, mini-batches, more epochs) and see how much of the gap to the exact solution closes.
- Try a non-linear scorer (e.g. a small neural network or gradient-boosted trees) with a ranking objective.
- Implement LambdaRank, which weights each pair by its effect on NDCG.